# Atividade Prática — Aula 6: Visualização Interativa com Plotly

Esta atividade foi construída com base nos slides da Aula 6, que apresentam a transição do gráfico estático para o gráfico como **aplicativo de exploração**, com foco em **hover**, **zoom**, **pan**, **filtros visuais**, **Plotly Express** e construção de componentes que mais tarde podem virar um dashboard. fileciteturn7file0

## Ideia central da aula
A interatividade existe para apoiar a investigação do gestor em tempo real.  
Mesmo assim, a aula reforça uma regra essencial: **interatividade não substitui clareza**. O gráfico precisa continuar limpo, bem titulado e orientado à decisão. fileciteturn7file0

## Regras da atividade
- O notebook orienta, mas **você deve construir os códigos**.
- Use **Plotly Express** sempre que possível.
- Após cada visual principal, escreva uma breve **interpretação humana**.
- Teste hover, zoom e isolamento por legenda antes de concluir.
- Pense como desenvolvedor: os gráficos de hoje poderão virar o dashboard de amanhã. fileciteturn7file0

## Dataset da atividade
Arquivo: `vendas_brasil_clean_aula6_plotly.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias.

**Sugestão:**
- `pandas`
- `plotly.express`
- `plotly.graph_objects` (opcional)


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## 2. Leitura e inspeção inicial da base

Leia o arquivo `vendas_brasil_clean_aula6_plotly.csv` em um DataFrame chamado `df`.

Depois:
1. exiba as primeiras linhas
2. verifique o tamanho da base
3. confira os tipos das colunas
4. confirme se `data_venda` está em formato adequado para análises temporais
5. identifique quais colunas podem alimentar:
   - comparação de categorias
   - evolução no tempo
   - relação entre variáveis
   - distribuição espacial


In [ ]:
# Leitura e inspeção inicial
arquivo = "vendas_brasil_clean_aula6_plotly.csv"
df = pd.read_csv(arquivo)

# Garantir tipagem temporal para análises de série
df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")

print("Primeiras linhas:")
display(df.head())

print(f"\nTamanho da base: {df.shape[0]} linhas x {df.shape[1]} colunas")
print("\nTipos das colunas:")
print(df.dtypes)

print("\n`data_venda` em datetime?", pd.api.types.is_datetime64_any_dtype(df["data_venda"]))
print("Valores nulos em `data_venda`:", df["data_venda"].isna().sum())

print("\nColunas úteis por objetivo analítico:")
print("- Comparação de categorias: categoria, canal_venda, uf")
print("- Evolução no tempo: data_venda, receita, lucro, quantidade")
print("- Relação entre variáveis: receita, lucro, quantidade, desconto, margem_lucro")
print("- Distribuição espacial: latitude, longitude, uf, cidade")

## 3. Matriz de decisão do analista

### Respostas
1. Esta atividade faz mais sentido em Plotly porque os gráficos exigem exploração interativa (hover para contexto adicional, zoom para sazonalidade e filtros por legenda/categoria), algo essencial para perguntas em tempo real.
2. Eu preferiria gráfico estático quando o objetivo for publicação em relatório PDF/impresso com leitura rápida e sem necessidade de investigação dinâmica.
3. O destino desta análise é claramente mais **tela/web**, pois o valor está na navegação interativa e no potencial de reaproveitar os visuais em dashboard.

## 4. Missão 1 — Barras: Receita por Canal

A missão prática da aula pede construir um gráfico de barras para responder:
**qual canal vende mais?**  
Os slides também sugerem usar o hover para injetar métricas secundárias sem poluir a tela. fileciteturn7file0

### Tarefa
1. Agregue a `receita` por `canal_venda`
2. Inclua também uma métrica secundária, como `quantidade`
3. Construa um gráfico de barras com Plotly Express
4. Ordene do maior para o menor
5. Faça o hover mostrar mais do que apenas a receita

### Perguntas
- Qual canal lidera a receita?
- O canal líder também lidera em quantidade?
- O hover ajudou a enriquecer a leitura sem poluir a tela?


In [ ]:
# Receita por canal com hover enriquecido
canal_resumo = (
    df.groupby("canal_venda", as_index=False)
      .agg(receita_total=("receita", "sum"), quantidade_total=("quantidade", "sum"))
      .sort_values("receita_total", ascending=False)
)

fig_receita_canal = px.bar(
    canal_resumo,
    x="canal_venda",
    y="receita_total",
    text_auto=True,
    hover_data={"quantidade_total": True, "receita_total": ":,.2f"},
    title="Receita total por canal de venda"
)

fig_receita_canal.update_layout(
    xaxis_title="Canal de venda",
    yaxis_title="Receita (R$)",
    yaxis_tickprefix="R$ "
)

display(canal_resumo)
fig_receita_canal.show()

### Insight obrigatório
O canal **Online** lidera a receita (R$ 893.354,53) e também lidera em quantidade (656), seguido por **App**.  
**Loja Física** aparece na última posição em receita e volume, indicando menor tração relativa.  
Como próximo passo, um gestor poderia investigar mix de produtos e ticket médio por canal para entender por que Online e App performam melhor.

## 5. Missão 2 — Linhas: Sazonalidade da Receita Mensal

Os slides destacam que o gráfico de linhas, com zoom e pan, é ideal para navegar no tempo e isolar picos. Também reforçam que o eixo temporal precisa estar corretamente tipado no Pandas. fileciteturn7file0

### Tarefa
1. Converta `data_venda` em data, se necessário
2. Agregue a `receita` por mês
3. Construa um gráfico de linha interativo
4. Teste o zoom nos meses de pico
5. Observe se novembro e dezembro se destacam

### Perguntas
- Existe sazonalidade?
- Quais meses chamam atenção?
- O zoom ajudou a explorar melhor a parte final da série?


In [ ]:
# Receita mensal e sazonalidade
df_mensal = (
    df.dropna(subset=["data_venda"])
      .assign(mes=lambda x: x["data_venda"].dt.to_period("M").dt.to_timestamp())
      .groupby("mes", as_index=False)["receita"].sum()
      .sort_values("mes")
)

fig_receita_mensal = px.line(
    df_mensal,
    x="mes",
    y="receita",
    markers=True,
    title="Evolução da receita mensal"
)

fig_receita_mensal.update_layout(
    xaxis_title="Mês",
    yaxis_title="Receita (R$)",
    yaxis_tickprefix="R$ "
)

print("Top 5 meses por receita:")
display(df_mensal.sort_values("receita", ascending=False).head())
fig_receita_mensal.show()

### Insight obrigatório
A série mensal mostra tendência de aceleração no fim do ano, com picos claros em **novembro (R$ 428.009,19)** e **dezembro (R$ 417.833,36)**.  
Isso caracteriza sazonalidade forte no último bimestre, possivelmente ligada a Black Friday e compras de fim de ano.  
O zoom ajuda a comparar melhor esses meses finais com os demais períodos da série.

## 6. Missão 3 — Dispersão: Lucro vs. Receita segmentado por Categoria

Os slides mostram o uso do gráfico de dispersão para revelar correlação entre KPIs e identificar anomalias, com grande apoio do hover. fileciteturn7file0

### Tarefa
1. Construa um scatter plot com:
   - eixo X = `receita`
   - eixo Y = `lucro`
2. Use `categoria` como cor
3. Inclua no hover:
   - `produto`
   - `canal_venda`
   - `uf`
   - `margem_lucro`
4. Tente identificar algum ponto fora da curva

### Perguntas
- Existe correlação visual entre receita e lucro?
- Há anomalias?
- O hover ajuda a transformar um ponto em um caso investigável?


In [ ]:
# Dispersão: lucro vs receita por categoria
if "margem_lucro" not in df.columns:
    df["margem_lucro"] = (df["lucro"] / df["receita"]).replace([float("inf"), -float("inf")], pd.NA)

fig_lucro_receita = px.scatter(
    df,
    x="receita",
    y="lucro",
    color="categoria",
    hover_data=["produto", "canal_venda", "uf", "margem_lucro"],
    title="Relação entre receita e lucro por categoria",
    opacity=0.75
)

fig_lucro_receita.update_layout(
    xaxis_title="Receita (R$)",
    yaxis_title="Lucro (R$)",
    legend_title="Categoria"
)

correlacao = df[["receita", "lucro"]].corr().iloc[0, 1]
print(f"Correlação receita x lucro: {correlacao:.3f}")

# Pontos potenciais fora da curva: alta receita com margem baixa
limite_receita_alta = df["receita"].quantile(0.90)
limite_margem_baixa = df["margem_lucro"].quantile(0.10)
possiveis_anomalias = df[(df["receita"] >= limite_receita_alta) & (df["margem_lucro"] <= limite_margem_baixa)]

print("Possíveis anomalias (alta receita com margem_lucro baixa):")
display(possiveis_anomalias[["produto", "categoria", "canal_venda", "uf", "receita", "lucro", "margem_lucro"]].head(10))

fig_lucro_receita.show()

### Insight obrigatório
A correlação entre receita e lucro é **positiva e forte** (≈ 0,915), indicando que vendas maiores tendem a gerar mais lucro.  
Os pontos fora do padrão aparecem em vendas de alta receita com margem baixa (ex.: alguns registros de **Notebook Pro**, incluindo um caso com margem ~8,44%).  
Uma ação analítica seguinte é investigar política de desconto, custos e canal desses casos para proteger margem sem perder volume.

## 7. Exploração espacial — Onde está concentrada a operação?

Os slides incluem mapas geográficos como resposta para perguntas espaciais como:
**onde está concentrada nossa operação?** fileciteturn7file0

### Tarefa
Usando `latitude` e `longitude`, crie um mapa interativo que ajude a visualizar a operação.

### Sugestões
- use tamanho ou cor para uma métrica relevante (como receita)
- teste o zoom do mapa
- observe se há concentração regional

### Perguntas
- Onde a operação parece mais concentrada?
- O mapa ajudou mais do que uma simples tabela por UF?


In [ ]:
# Mapa interativo da operação
mapa_df = (
    df.groupby(["uf", "latitude", "longitude"], as_index=False)
      .agg(receita_total=("receita", "sum"), quantidade_total=("quantidade", "sum"))
)

resumo_uf = (
    df.groupby("uf", as_index=False)["receita"]
      .sum()
      .sort_values("receita", ascending=False)
)

print("Top 5 UFs por receita:")
display(resumo_uf.head())

fig_mapa_operacao = px.scatter_map(
    mapa_df,
    lat="latitude",
    lon="longitude",
    color="receita_total",
    size="quantidade_total",
    hover_name="uf",
    hover_data={"receita_total": ":,.2f", "quantidade_total": True},
    zoom=3,
    title="Concentração geográfica da operação"
)

fig_mapa_operacao.update_layout(margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig_mapa_operacao.show()

## 8. A tríade da interatividade

Escolhendo o gráfico de linha da receita mensal:

1. **Hover** acrescenta detalhes exatos de cada mês (valor de receita), sem carregar o gráfico com rótulos em todos os pontos.
2. **Zoom** melhora a investigação ao permitir foco em janelas específicas, como o fim do ano, onde há picos relevantes.
3. **Legenda/filtros visuais** (quando há múltiplas séries) ajudam a isolar categorias/canais para comparação direta sem criar gráficos separados.

## 9. Plotly Express — Máximo impacto, mínimo código

1. Dizer que Plotly Express é **declarativo** significa descrever o que quero visualizar (eixo X, Y, cor, tamanho, hover) sem programar manualmente cada elemento gráfico.
2. Isso facilita a vida do analista porque reduz código repetitivo, acelera prototipação e mantém integração natural com DataFrame do Pandas.
3. Essa abordagem conecta diretamente com dashboards em Streamlit: o mesmo objeto `fig` pode ser reaproveitado com poucas adaptações na aplicação web.

## 10. Clareza continua obrigatória

### O que foi melhorado no gráfico revisado
1. Título mais específico: foco em receita por canal e ordenação.
2. Eixos renomeados com unidade monetária (`R$`).
3. Hover enriquecido com quantidade total além da receita.
4. Ordenação visual e rótulos de valor para leitura imediata.

### Resultado
Sim, o gráfico ficou mais claro: a hierarquia entre canais aparece rapidamente e o hover adiciona contexto sem poluição visual.

In [ ]:
# Versão revisada com foco em clareza (Missão 10)
fig_receita_canal_claro = px.bar(
    canal_resumo,
    x="receita_total",
    y="canal_venda",
    orientation="h",
    color="receita_total",
    text="receita_total",
    hover_data={"quantidade_total": True, "receita_total": ":,.2f"},
    title="Receita por canal de venda (ordem decrescente e hover com quantidade)"
)

fig_receita_canal_claro.update_traces(texttemplate="R$ %{text:,.0f}", textposition="outside")
fig_receita_canal_claro.update_layout(
    xaxis_title="Receita total (R$)",
    yaxis_title="Canal de venda",
    legend_title="Receita",
    coloraxis_colorbar_title="Receita"
)

fig_receita_canal_claro.show()

## 11. O gráfico não fala sozinho

### Gráfico 1 — Receita por canal
- **Insight principal:** Online e App concentram maior receita, com liderança do Online.
- **Decisão gerencial:** priorizar investimento comercial e digital nos canais de maior retorno.
- **Pergunta seguinte:** qual canal combina melhor receita com margem por categoria?

### Gráfico 2 — Receita mensal
- **Insight principal:** há sazonalidade forte no fim do ano (novembro e dezembro).
- **Decisão gerencial:** antecipar estoque, marketing e capacidade operacional para Q4.
- **Pergunta seguinte:** quais categorias puxam os picos de novembro/dezembro?

## 12. Missão prática final — Mapeando e construindo

### 1) Barras — Receita por canal
- **Pergunta de negócio:** qual canal vende mais?
- **Colunas necessárias:** `canal_venda`, `receita`, `quantidade`.
- **Interpretação humana:** Online lidera em receita e quantidade; Loja Física fica atrás.

### 2) Linhas — Sazonalidade da receita mensal
- **Pergunta de negócio:** como a receita evolui ao longo do tempo e onde estão os picos?
- **Colunas necessárias:** `data_venda`, `receita`.
- **Interpretação humana:** há sazonalidade com destaque para novembro e dezembro.

### 3) Dispersão — Lucro vs Receita por categoria
- **Pergunta de negócio:** receita maior está associada a lucro maior e existem anomalias?
- **Colunas necessárias:** `receita`, `lucro`, `categoria`, `produto`, `canal_venda`, `uf`, `margem_lucro`.
- **Interpretação humana:** correlação positiva forte (0,915), com casos de alta receita e margem baixa que exigem investigação.

### 4) Mapa — Concentração geográfica da operação
- **Pergunta de negócio:** onde a operação está mais concentrada?
- **Colunas necessárias:** `latitude`, `longitude`, `uf`, `receita`, `quantidade`.
- **Interpretação humana:** há concentração relevante em UFs como MG e SP, seguida por CE, SC e PR.

## 13. Mindset de desenvolvedor

1. Eu organizaria objetos com padrão claro, por exemplo: `fig_receita_canal`, `fig_receita_mensal`, `fig_lucro_receita`, `fig_mapa_operacao`.
2. Partes reaproveitáveis para web: leitura/tratamento de dados, funções de agregação e geração dos objetos `fig`.
3. Vale padronizar desde já: nomes de variáveis, títulos e eixos, formatação monetária, e estrutura de código por bloco (preparo → agregação → visualização → insight).

## 14. Desafio extra (opcional)

### Interpretação final
No gráfico de receita por UF, **MG** e **SP** aparecem como maiores contribuintes, com **CE**, **SC** e **PR** logo na sequência.  
Esse recorte ajuda a priorizar ações comerciais por território e pode orientar metas regionais.  
Uma próxima pergunta útil seria: a liderança por UF vem de maior volume, maior ticket médio ou melhor mix de produtos?

In [ ]:
# Desafio extra: receita por UF (barras horizontais)
receita_uf = (
    df.groupby("uf", as_index=False)["receita"]
      .sum()
      .sort_values("receita", ascending=True)
)

display(receita_uf.tail(5).sort_values("receita", ascending=False))

fig_receita_uf = px.bar(
    receita_uf,
    x="receita",
    y="uf",
    orientation="h",
    title="Receita total por UF",
    hover_data={"receita": ":,.2f"}
)

fig_receita_uf.update_layout(
    xaxis_title="Receita (R$)",
    yaxis_title="UF",
    xaxis_tickprefix="R$ "
)

fig_receita_uf.show()

## 15. Entrega esperada

Seu notebook deve demonstrar:
- uso correto de Plotly Express
- escolha adequada do gráfico para a pergunta
- exploração consciente de hover, zoom, pan e legenda
- compromisso com clareza
- interpretação textual consistente

### Mensagem principal da aula
A tecnologia explora.  
Mas só o analista transforma exploração em decisão. fileciteturn7file0
